# HW7: Beam Search Decoding - News Headline Generation

In this exercise, you are going to learn and implement decoding techniques for sequence generation. Usually, the sequence is generated word-by-word from a model. In each step, the model predicted the most likely word based on the predicted words in previous steps (this is called auto-regressive decoding).

As such, it is very important how you decide on what to predicted at each step, as it will be conditioned on to predicted all of the following steps. We will implement two of main decoding techniques introduced in the lecture: **Greedy Decoding** and **Beam Search Decoding**. Greedy Decoding immediately chooses the word with best score at each step, while Beam Search Decoding focuses on the sequence that give the best score overall.

To complete this exercise, you will need to complete the methods for decoding for a text generation model trained on [New York Times Comments and Headlines dataset](https://www.kaggle.com/aashita/nyt-comments). The model is trained to predict a headline for the news given seed text. You do not need to train any model model in this exercise as we provide both the pretrained model and dictionary.


## Download model and vocab and setup

In [1]:
!wget -O vocab.txt https://www.dropbox.com/s/ht12ua9vpkep6l8/hw9_vocab.txt?dl=0
!wget -O model.bin https://www.dropbox.com/s/okmri7cnd729rr5/hw9_model.bin?dl=0

--2025-02-25 14:50:58--  https://www.dropbox.com/s/ht12ua9vpkep6l8/hw9_vocab.txt?dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/zlkw3il9cj4c121vtrrxh/hw9_vocab.txt?rlkey=m5gflik1lhvpenwydd3gfwvp7&dl=0 [following]
--2025-02-25 14:50:58--  https://www.dropbox.com/scl/fi/zlkw3il9cj4c121vtrrxh/hw9_vocab.txt?rlkey=m5gflik1lhvpenwydd3gfwvp7&dl=0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc9dcc2aeaa47ae406fc80ac0aaa.dl.dropboxusercontent.com/cd/0/inline/CkxDAIUbmGS_ahNGJXKaD1PhrmAVqBsVzt3iApmoNK2n2anirQK_nXpdNYdqa5f7NWCYLWov_NvfX_T99uqmiOFX-j0JG7chNznLI12zbPPfpRGr1uSebXSl8gGEgW-JW6o0uqSwPDJRcMtnhaZutfiP/file# [following]
--2025-02-25 14:50:59--  https://uc9dcc2aeaa47ae406fc80ac0aaa.dl.dropboxusercontent

In [2]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

In [14]:
import torch.nn.functional as F

In [15]:
class RNNmodel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, dropout_rate):

        super().__init__()
        self.embedding_dim = embedding_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, 128, num_layers=2,
                     batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(128, vocab_size)

    def forward(self, src):
        embedding = self.embedding(src)
        output,_ = self.rnn(embedding)
        output = self.dropout(output)
        prediction = self.fc2(output)
        return prediction

In [16]:
with open("vocab.txt") as f:
  vocab_file = f.readlines()
embedding_dim = 64
dropout_rate = 0.2

model = RNNmodel(len(vocab_file), embedding_dim, dropout_rate)
model.load_state_dict(torch.load("model.bin",map_location='cpu'))
model.eval()

<ipython-input-16-e56f35018f72>:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model.bin",map_location='cpu'))


RNNmodel(
  (embedding): Embedding(10054, 64)
  (rnn): LSTM(64, 128, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=128, out_features=10054, bias=True)
)

In [17]:
vocab = [v.strip() for v in vocab_file]
vocab_size = len(vocab)
print(f"Vocab Size: {vocab_size}")
vocab[:10]

Vocab Size: 10054


['<unk>', '<pad>', '<eos>', 'the', 'a', 'to', 'of', 's', 'in', 'for']

In [18]:
stoi = { ch:i for i,ch in enumerate(vocab) }
tokenizer = Tokenizer(WordLevel(stoi, unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()
tokenized_text = tokenizer.encode("the a of to unknowns")
print(tokenized_text)
print(tokenized_text.ids)
print(tokenized_text.tokens)
print(tokenizer.decode(tokenized_text.ids))

Encoding(num_tokens=5, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
[3, 4, 6, 5, 0]
['the', 'a', 'of', 'to', '<unk>']
the a of to <unk>


## 1. TODO: Greedy decode
Normally, in sequence generation task, the model will continue generating tokens until an end-of-sequence symbol appear or the maximum length is reached. For this task:
- The end-of-sequence symbol is "< eos >" and its index is 2
- Use the maximum generation length of 15

In [19]:
eos_token = '<eos>'
max_gen_length = 15

In [20]:
def greedy_decode(seed_text, tokenizer):
    """Greedy decodes with seed text.

        Args:
        seed_text: The seed string to be used as initial input to the model.
        tokenizer: The tokenizer for converting word to index and back.

        Your code should do the followings:
          1. Convert current_text to sequences of indices
          2. Predict the next token using the model and choose the token with the highest score as output
          3. Append the predicted index to current_text
          4. Loop until completion
          5. Return text prediction and a list of probabilities of each step

        You do not need to stop early when end-of-sequence token is generated and can continue decoding
        until max_gen_length is reached. We can filter the eos token out later.
    """
    output = seed_text
    probs = []

    for _ in range(max_gen_length):
        tokenized_input = tokenizer.encode(output)
        input_ids = torch.tensor([tokenized_input.ids])

        with torch.no_grad():
            logits = model(input_ids)[0, -1, :]

        probabilities = F.softmax(logits, dim=0)
        best_index = torch.argmax(probabilities).item()
        best_token = vocab[best_index]

        if best_token == '<eos>':
            break

        probs.append(probabilities[best_index].item())
        output += " " + best_token


    return output,probs

In [21]:
def clean_output(text, eos_token):
    """Drop eos_token and every words that follow"""
    words = text.split()
    if eos_token in words:
        return " ".join(words[:words.index(eos_token)])
    return text

In [23]:
sample_seeds = ["to", "america", "people", "next", "picture", "on"]
for seed in sample_seeds:
    output, probs = greedy_decode(seed, tokenizer)
    output_text = clean_output(output, eos_token)
    print(output_text)
    print(probs)

to encourage creativity in the new york bill
[0.04071051999926567, 0.2661305069923401, 0.2581596076488495, 0.23116469383239746, 0.07794320583343506, 0.06978005170822144, 0.22445209324359894]
america s lethal export
[0.44965454936027527, 0.04639975354075432, 0.7206951975822449]
people to balloon to make a criminal with a dog with a callous rival
[0.35848501324653625, 0.039465807378292084, 0.2695298492908478, 0.03377285599708557, 0.28156715631484985, 0.030896684154868126, 0.3262556791305542, 0.32244065403938293, 0.03489116579294205, 0.22298365831375122, 0.5808284282684326, 0.04674507677555084, 0.26616621017456055]
next phenom english clubs 2 call another deal in the same arrivals
[0.17242352664470673, 0.2323564738035202, 0.7396267056465149, 0.0788838341832161, 0.04404299706220627, 0.0576934739947319, 0.03296951204538345, 0.33040091395378113, 0.6040376424789429, 0.03174588084220886, 0.07468773424625397]
picture perfect chapter a spot of view of banning care
[0.25000014901161194, 0.0176074

Your output should be:

*   to encourage creativity in the new york bill
*   america s lethal export
*   people to balloon to make a criminal with a dog with a callous rival
*   next phenom english clubs 2 call another deal in the same arrivals
*   picture perfect chapter a spot of view of banning care  
*   on the catwalk in saudi arabia







## 2. TODO: Beam search decode

Another well-known decoding method is beam search decoding that focuses more on the overall sequence score.

Instead of greedily choosing the token with the highest score for each step, beam search decoding expands all possible next tokens and keeps the __k__ most likely sequence at each step, where __k__ is a user-specified beam size. A sequence score is also calculated according user-specified cal_score() function.
The beam with the highest score after the decoding process is done will be the output.

There are a few things that you need to know before implementing a beam search decoder:
- When the eos token is produced, you can stop expanding that beam
- However, the ended beams must be sorted together with active beams
- The decoding ends when every beams are either ended or reached the maximum length, but for this task, you can continue decoding until the max_gen_len is reached
- We usually work with probability in log scale to avoid numerical underflow. You should use np.log(score) before any calculation
- **As probabilities for some classes will be very small, you must add a very small value to the score before taking log e.g np.log(prob + 0.00000001)**

#### Sequence Score
The naive way to calculate the sequence score is to __multiply every token scores__ together. However, doing so will make the decoder prefer shorter sequence as you multiply the sequence score with a value between \[0,1\] for every tokens in the sequence. Thus, we usually normalize the sequence score with its length by calculating its __geometric mean__ instead.

**You should do this in log scale**

In [24]:
import numpy as np

In [25]:
def cal_score(score_list, length, normalized=False): #cal score for each beam from a list of probs
    log_scores = np.log(np.array(score_list) + 1e-8)
    if normalized:
        score = log_scores.sum() / length
    else:
        score = log_scores.sum()
    return score

In [46]:
def beam_search_decode(seed_text, max_gen_len, tokenizer, beam_size=5, normalized=False):
    """We will do beam search decoing using seed text in this function.

    Output:
    beams: A list of top k beams after the decoding ended, each beam is a list of
      [seed_text, list of scores, length]

    Your code should do the followings:
    1.Loop until max_gen_len is reached.
    2.During each step, loop thorugh each beam and use it to predict the next word.
      If a beam is already ended, continues without expanding.
    3.Sort all hypotheses according to cal_score().
    4.Keep top k hypotheses to be used at the next step.
    """
    # For each beam we will store (generated text, list of scores, and current length, is_finished)
    # Add initial beam
    # Convert seed_text (list of ints) to tensor

    tokenized_text = tokenizer.encode(seed_text).ids
    # beams: each beam is [current_text_tensor, score_list, length, is_finished]
    beams = [[torch.tensor([tokenized_text]), [], 0, False]]

    for _ in range(max_gen_len):
        hypo = []

        for beam in beams:
            current_text, score_list, length, is_finished = beam
            if is_finished:
                # If the beam is finished, just carry it over without expanding
                hypo.append(beam)
                continue

            with torch.no_grad():
                # current_text is a tensor of shape [1, current_length]
                logits = model(current_text)[0, -1, :]  # shape: [vocab_size]

            probabilities = F.softmax(logits, dim=0)
            topk_probs, topk_indices = torch.topk(probabilities, beam_size)

            for prob, predicted_index in zip(topk_probs, topk_indices):
                # predicted_index is a single token ID
                predicted_index = predicted_index.unsqueeze(0).unsqueeze(0)  # shape [1,1]

                # Concatenate along dim=1
                new_text = torch.cat([current_text, predicted_index], dim=1)  # shape [1, current_length+1]
                new_length = length + 1

                # Check end-of-sequence condition
                is_finished_new = (predicted_index.item() == stoi[eos_token])

                new_score_list = score_list + [prob.item() + 1e-8]
                hypo.append([new_text, new_score_list, new_length, is_finished_new])

        # Sort and keep top k
        hypo.sort(key=lambda hyp: cal_score(hyp[1], hyp[2], normalized), reverse=True)
        beams = hypo[:beam_size]

    return beams

## 3. Generate!
Generate 6 sentences based on the given seed texts.

Decode with the provided seed texts with beam_size 5, max_gen_len 10. Compare the results between greedy, normalized, and unnormalized decoding.

Print the result using greedy decoding and top 2 results each using unnormalized and normalized decoing for each seed text.

Also, print scores of each candidate according to cal_score(). Use normalization for greedy decoding.

In [54]:
sample_seeds = ["to", "america", "people", "next", "picture", "on"]
max_gen_len=10
for seed in sample_seeds:
    print("-Greedy-")
    output, probs = greedy_decode(seed, tokenizer)
    output_text = clean_output(output, eos_token)
    score = cal_score(probs, len(probs), normalized=True)
    print(f"{output_text} {round(np.exp(score),2)}")

    print("-Unnormalized")
    beams = beam_search_decode(seed, max_gen_len, tokenizer, beam_size=5, normalized=False)
    for beam in beams[:2]:
        output = tokenizer.decode(beam[0].squeeze().tolist())
        output_text = clean_output(output, eos_token)  # Clean the output
        if beam[3]:
            score = cal_score(beam[1][:-1], beam[2]-1, normalized=False)
        else:
            score = cal_score(beam[1], beam[2], normalized=False)
        print(f"{output_text.title()}  {round(np.exp(score), 2)}")

    print("-Normalized-")
    beams = beam_search_decode(seed, max_gen_length, tokenizer, beam_size=5, normalized=True)
    for beam in beams[:2]:
        output_text = tokenizer.decode(beam[0].squeeze().tolist())
        cleaned_text = clean_output(output_text, eos_token)  # Clean the output
        if beam[3]:
            score = cal_score(beam[1][:-1], beam[2]-1, normalized=True)
        else:
            score = cal_score(beam[1], beam[2], normalized=True)

        print(f"{cleaned_text.title()}  {round(np.exp(score), 2)}")

    print(" ")


-Greedy-
to encourage creativity in the new york bill 0.13
-Unnormalized
To Consult Exploring Recipes For New Jersey  0.0
To Consult Exploring Recipes Up The Pacific Northwest  0.0
-Normalized-
To Consult Exploring Recipes Up The Pacific Northwest  0.17
To Consult Exploring Recipes Up The Least Of The Week  0.16
 
-Greedy-
america s lethal export 0.25
-Unnormalized
America S Lethal Export  0.02
America S Desert Aisles  0.01
-Normalized-
America S Lethal Export  0.25
America S Desert Aisles  0.2
 
-Greedy-
people to balloon to make a criminal with a dog with a callous rival 0.14
-Unnormalized
People To Balloon For A Criminal  0.0
People To Balloon For A Criminal With Trump  0.0
-Normalized-
People To Balloon For A Criminal With A Second Fiddle  0.13
People To Balloon For A Criminal With Trump  0.13
 
-Greedy-
next phenom english clubs 2 call another deal in the same arrivals 0.12
-Unnormalized
Next S Blist Revue  0.0
Next Phenom English Clubs 1 A Chance To Be Back  0.0
-Normalized-
Next

Your output should be:


```
-Greedy-
to encourage creativity in the new york bill  0.13
-Unnormalized-
To Consult Exploring Recipes For New Jersey 0.00
To Consult Exploring Recipes Up The Pacific Northwest 0.00
-Normalized-
To Consult Exploring Recipes Up The Pacific Northwest 0.17
To Consult Exploring Recipes Up The Least Of The Week 0.16

-Greedy-
america s lethal export  0.25
-Unnormalized-
America S Lethal Export 0.02
America S Desert Aisles 0.01
-Normalized-
America S Lethal Export 0.25
America S Desert Aisles 0.20

-Greedy-
people to balloon to make a criminal with a dog with a callous rival  0.14
-Unnormalized-
People To Balloon For A Criminal 0.00
People To Balloon For A Criminal With Trump 0.00
-Normalized-
People To Balloon For A Criminal With A Second Fiddle 0.13
People To Balloon For A Criminal With Trump 0.13

-Greedy-
next phenom english clubs 2 call another deal in the same arrivals  0.12
-Unnormalized-
Next S Blist Revue 0.00
Next Phenom English Clubs 1 A Chance To Be Back 0.00
-Normalized-
Next S Blist Revue 0.14
Next Phenom English Clubs 1 A Chance To Be Back 0.14

-Greedy-
picture perfect chapter a spot of view of banning care  0.08
-Unnormalized-
Picture Perfect Use Coffee 0.00
Picture Korean A Bonanza Of Pancakes 0.00
-Normalized-
Picture Korean A Bonanza Of Contemplation Times Of Trump S Son 0.12
Picture Korean A Bonanza Of Pancakes 0.07

-Greedy-
on the catwalk in saudi arabia  0.19
-Unnormalized-
On The Billboard Chart 0.00
On The Catwalk In Saudi Arabia 0.00
-Normalized-
On The Whole30 Diet Vowing To Eat Smarter Carbs To Be 0.27
On The Whole30 Diet Vowing To Eat Smarter Carbs For Because 0.26

```



# Answer Questions in MyCourseVille!

Use the seed word "usa" to answer questions in MCV.

In [55]:
sample_seeds = ["usa"]
max_gen_len=10
for seed in sample_seeds:
    print("-Greedy-")
    output, probs = greedy_decode(seed, tokenizer)
    output_text = clean_output(output, eos_token)
    score = cal_score(probs, len(probs), normalized=True)
    print(f"{output_text} {round(np.exp(score),2)}")

    print("-Unnormalized")
    beams = beam_search_decode(seed, max_gen_len, tokenizer, beam_size=5, normalized=False)
    for beam in beams[:2]:
        output = tokenizer.decode(beam[0].squeeze().tolist())
        output_text = clean_output(output, eos_token)  # Clean the output
        if beam[3]:
            score = cal_score(beam[1][:-1], beam[2]-1, normalized=False)
        else:
            score = cal_score(beam[1], beam[2], normalized=False)
        print(f"{output_text.title()}  {round(np.exp(score), 2)}")

    print("-Normalized-")
    beams = beam_search_decode(seed, max_gen_length, tokenizer, beam_size=5, normalized=True)
    for beam in beams[:2]:
        output_text = tokenizer.decode(beam[0].squeeze().tolist())
        cleaned_text = clean_output(output_text, eos_token)  # Clean the output
        if beam[3]:
            score = cal_score(beam[1][:-1], beam[2]-1, normalized=True)
        else:
            score = cal_score(beam[1], beam[2], normalized=True)

        print(f"{cleaned_text.title()}  {round(np.exp(score), 2)}")

    print(" ")


-Greedy-
usa s duty to investigate 0.09
-Unnormalized
Usa S Duty To Investigate  0.0
Usa S Bleak Season 3 Episode 4 Recap Chicken  0.0
-Normalized-
Usa S Bleak Season 1 Episode 2 Darkness Descends  0.19
Usa S Bleak Season 3 Episode 4 Recap A Seduction  0.18
 
